In [2]:
# Get Dataset from Roboflow

In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="BVw8dJRkVeedL6yyW1bq")
project = rf.workspace("badminton-court").project("badminton-court-dataset")
version = project.version(2)
dataset = version.download("coco")
                

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 MB 14.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 941.9/941.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 14.1 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: idna╺━━━━━━━━━━━━━━━━━━━ 3/6 [opencv-python-headless]
    Found existing installation: idna 3.11m━━━━━━━━━━━━━━━━━━━ 3/6 [opencv-python-headless]
    Uninstalling idna-3.11:m╺━━━━━━━━━━━━━━━━━━━ 3/6 [opencv-python-headless]
      Successfully uninstalled idna-3.1190m━━━━━━━━━━━━━━━━━━━ 3/6 [opencv-python-headless]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [roboflow]5/6 [roboflow]-headless]
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to badminton-court-dataset-2 in coco:: 100%|██████████| 1015/1015 [00:00<00:00, 6603.47it/s]


In [11]:
import shutil

In [6]:
shutil.move("Data/badminton-court-dataset-2/train",
             "Data/train"
)
shutil.move("Data/badminton-court-dataset-2/valid",
            "Data/valid"
)
shutil.move("Data/badminton-court-dataset-2/test",
            "Data/test"
)
shutil.move("Data/badminton-court-dataset-2/README.dataset.txt",
            "Data/README.dataset.txt"
)
shutil.move("Data/badminton-court-dataset-2/README.roboflow.txt",
            "Data/README.roboflow.txt"
)

'Data/README.roboflow.txt'

# Install requirements

In [12]:
!pip install torch torchvision opencv-python

  Using cached numpy-2.2.6-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp313-cp313-macosx_14_0_arm64.whl (5.1 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.5
    Uninstalling numpy-2.3.5:
      Successfully uninstalled numpy-2.3.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.2.6 which is incompatible.


In [15]:
import torch
import torchvision
import cv2
print(torch.__version__)
print(torchvision.__version__)
print(cv2.__version__)

2.9.0
0.24.0
4.10.0


In [36]:
import importlib
import dataset
importlib.reload(dataset)
from dataset import get_transforms

In [37]:
import torchvision
from torchvision.models.detection.keypoint_rcnn import KeypointRCNNPredictor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_keypoints=30, num_classes=2):
    # Load a pretrained model (why: faster convergence, better features)
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(weights="DEFAULT")

    # Replace box predictor to match num_classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Replace keypoint predictor to match num_keypoints
    in_features_kp = model.roi_heads.keypoint_predictor.kps_score_lowres.in_channels
    model.roi_heads.keypoint_predictor = KeypointRCNNPredictor(in_features_kp, num_keypoints)

    return model


In [38]:
from torch.utils.data import DataLoader
from dataset import collate_fn

In [39]:
import torch
from torch.optim import SGD

def train_one_epoch(model, optimizer, loader, device, epoch):
    model.train()
    total = 0.0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total += loss.item()

    avg = total / max(len(loader), 1)
    print(f"Epoch {epoch}: avg_loss={avg:.4f}")


In [40]:
@torch.no_grad()
def validate(model, loader, device, score_thresh=0.5):
    model.eval()
    counts = []

    for images, _ in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for out in outputs:
            scores = out["scores"].detach().cpu()
            counts.append(int((scores >= score_thresh).sum().item()))

    print(f"Val: avg detections >= {score_thresh}: {sum(counts)/max(len(counts),1):.2f}")


In [41]:
import torch
from dataset import CourtKeypointsCoco, get_transforms, collate_fn

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # Update these paths to your dataset structure
    train_images = "/Users/nguyenanhvu/Documents/BadmintonAssistant/training/Data/train"
    train_ann = "/Users/nguyenanhvu/Documents/BadmintonAssistant/training/Data/train/_annotations.coco.json"
    val_images = "/Users/nguyenanhvu/Documents/BadmintonAssistant/training/Data/valid"
    val_ann = "/Users/nguyenanhvu/Documents/BadmintonAssistant/training/Data/valid/_annotations.coco.json"

    train_ds = CourtKeypointsCoco(train_images, train_ann, transforms=get_transforms(), num_keypoints=30)
    val_ds = CourtKeypointsCoco(val_images, val_ann, transforms=get_transforms(), num_keypoints=30)

    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=2, collate_fn=collate_fn)

    model = get_model(num_keypoints=30, num_classes=2).to(device)

    optimizer = SGD([p for p in model.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=0.0005)

    epochs = 20
    for epoch in range(1, epochs + 1):
        train_one_epoch(model, optimizer, train_loader, device, epoch)
        validate(model, val_loader, device, score_thresh=0.5)

    torch.save(model.state_dict(), "court_keypoints_kprcnn.pth")
    print("Saved model to court_keypoints_kprcnn.pth")

if __name__ == "__main__":
    main()

Device: cpu
Epoch 1: avg_loss=2.7790
Epoch 1: avg_loss=2.7790
Val: avg detections >= 0.5: 1.06
Val: avg detections >= 0.5: 1.06
Epoch 2: avg_loss=1.3027
Epoch 2: avg_loss=1.3027
Val: avg detections >= 0.5: 1.03
Val: avg detections >= 0.5: 1.03
Epoch 3: avg_loss=1.0647
Epoch 3: avg_loss=1.0647
Val: avg detections >= 0.5: 1.01
Val: avg detections >= 0.5: 1.01
Epoch 4: avg_loss=0.9457
Epoch 4: avg_loss=0.9457
Val: avg detections >= 0.5: 1.00
Val: avg detections >= 0.5: 1.00
Epoch 5: avg_loss=0.9178
Epoch 5: avg_loss=0.9178
Val: avg detections >= 0.5: 0.99
Val: avg detections >= 0.5: 0.99
Epoch 6: avg_loss=0.8241
Epoch 6: avg_loss=0.8241
Val: avg detections >= 0.5: 1.03
Val: avg detections >= 0.5: 1.03
Epoch 7: avg_loss=0.8118
Epoch 7: avg_loss=0.8118
Val: avg detections >= 0.5: 1.01
Val: avg detections >= 0.5: 1.01
Epoch 8: avg_loss=0.7968
Epoch 8: avg_loss=0.7968
Val: avg detections >= 0.5: 0.99
Val: avg detections >= 0.5: 0.99
Epoch 9: avg_loss=0.8004
Epoch 9: avg_loss=0.8004
Val: avg d